In [87]:
pip install habanero

Note: you may need to restart the kernel to use updated packages.


In [88]:
pip install -q keybert sentence-transformers hdbscan tqdm pandas

Note: you may need to restart the kernel to use updated packages.


In [89]:
pip install numpy


Note: you may need to restart the kernel to use updated packages.


In [90]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn


In [91]:
print(sys.executable)


/Users/ivar/Documents/VSCode/means-matter/.venv/bin/python


In [92]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [93]:
paper_list = pd.read_csv('db_enhance.csv')
paper_list['DOI'] = None


In [94]:
paper_list['link']
paper_list['link'][1]


'https://onlinelibrary.wiley.com/doi/10.1111/bioe.12512'

In [95]:
from tqdm import tqdm

In [96]:
from habanero import Crossref
import re

cr = Crossref()

# Example URL
url = "https://www.nature.com/articles/s41586-020-2649-2"

# Try to extract DOI pattern from URL
match = re.search(r"10\.\d{4,9}/[-._;()/:A-Z0-9]+", url, re.I)
if match:
    doi = match.group(0)
else:
    # If not in URL, you may need to query CrossRef by title
    title = "Array programming with NumPy"
    result = cr.works(query=title)
    doi = result['message']['items'][0]['DOI']

print(doi)


10.14293/s2199-1006.1.sor-uncat.a7759461.v1.raugfi


In [97]:
for idx, url in tqdm(paper_list['link'].items()):
    doi = None
    
    # --- Step 1: Try to extract DOI directly from URL ---
    match = re.search(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', url, re.I)
    if match:
        doi = match.group(0)
    
    # --- Step 2: If no DOI found, try to query CrossRef by article title if available ---
    # Optional: if you have titles, use them:
    title = paper_list.loc[idx, 'title']
    if not doi and pd.notna(title):
        try:
            result = cr.works(query=title)
            doi = result['message']['items'][0]['DOI']
        except:
            doi = None
    
    # Save DOI to DataFrame
    paper_list.at[idx, 'DOI'] = doi


1360it [05:12,  4.35it/s]


In [ ]:
paper_list.head()

,Unnamed: 0,author,title,sum,Enhancement,Germline,Ecology,Genetics,Offspring,journal,year,Type,link,wordcount,DOI,abstract
0,19197,Ezio Di Nucci,Fathers and Abortion,0.150,0.0,0.000,0.000,0.000,0.150,JMP,2014,Article,https://academic.oup.com/jmp/jmp/article/39/4/...,6465,10.2139/ssrn.1930829,None
1,6581,Christopher Kaczor,Ectogenesis and a right to the death of the pr...,0.151,0.0,0.008,0.000,0.143,0.000,Bioethics,2018,RESPONSE,https://onlinelibrary.wiley.com/doi/10.1111/bi...,2971,10.1111/bioe.12512,AbstractBoth many critics of abortion and many...
2,14586,"Bartha Maria Knoppers, Jennifer R Harris, Paul...",From genomic databases to translation: a call ...,0.151,0.0,0.000,0.000,0.147,0.003,JME,2011,Brief report,https://jme.bmj.com/content/37/8/515,687,10.1136/jme.2011.043042,The rapid rise of international collaborative ...
3,7607,GARRETT HARDIN,"When Technology Alters Nature, Two Kinds of Re...",0.151,0.0,0.000,0.151,0.000,0.000,HCR,1984,Features: Reviews,https://onlinelibrary.wiley.com/doi/10.2307/35...,3185,10.2307/3561751,None
4,8331,Jeffrey R. Botkin,Fetal Privacy and Confidentiality,0.151,0.0,0.000,0.000,0.048,0.104,HCR,1995,Article,https://onlinelibrary.wiley.com/doi/10.2307/35...,6759,10.2307/3562793,None


In [ ]:
# Prepare a new column for abstracts
paper_list['abstract'] = None

# Iterate over DOIs
for idx, doi in tqdm(paper_list['DOI'].items()):
    if pd.notna(doi):
        try:
            result = cr.works(ids=doi)
            abstract = result['message'].get('abstract', None)
            
            if abstract:
                # CrossRef abstracts are often XML-ish; strip tags
                abstract = re.sub(r"<[^>]+>", "", abstract)
            
            paper_list.at[idx, 'abstract'] = abstract
        except Exception as e:
            print(f"Failed to fetch DOI {doi}: {e}")
            paper_list.at[idx, 'abstract'] = None
    else:
        paper_list.at[idx, 'abstract'] = None

377it [37:44, 618.26s/it]

Failed to fetch DOI 10.1162/15265160360706435: ('Connection aborted.', TimeoutError(60, 'Operation timed out'))


624it [56:51, 305.09s/it]

Failed to fetch DOI 10.1007/s11019-017-9812-z: ('Connection aborted.', TimeoutError(60, 'Operation timed out'))


1360it [1:03:12,  2.79s/it]


In [ ]:
# Save the updated CSV
paper_list.to_csv("paper_list_with_abstracts.csv", index=False)

# Preview
print(paper_list[['link','DOI','abstract']].head())

                                                link                      DOI  \
0  https://academic.oup.com/jmp/jmp/article/39/4/...     10.2139/ssrn.1930829   
1  https://onlinelibrary.wiley.com/doi/10.1111/bi...       10.1111/bioe.12512   
2               https://jme.bmj.com/content/37/8/515  10.1136/jme.2011.043042   
3  https://onlinelibrary.wiley.com/doi/10.2307/35...          10.2307/3561751   
4  https://onlinelibrary.wiley.com/doi/10.2307/35...          10.2307/3562793   

                                            abstract  
0                                               None  
1  AbstractBoth many critics of abortion and many...  
2  The rapid rise of international collaborative ...  
3                                               None  
4                                               None  


In [ ]:
# Load CSV
df = pd.read_csv("paper_list_with_abstracts.csv")


Number of abstracts available for modeling: 424


['AbstractBoth many critics of abortion and many defenders of abortion have suggested that artificial wombs could end the abortion debate. If the fetus is removed from the uterus, women have an end to an unwanted pregnancy. If the living fetus is then put in an artificial uterus for ectogenesis, there is no termination of the life of the fetus. Joona Räsänen challenges this view in his article, Ectogenesis, abortion and a right to the death of the fetus. Räsänen provides three arguments for a right to secure the death of the human being in utero, namely the ‘right not to become a biological parent argument’, the ‘right to genetic privacy argument’, and the ‘right to property argument’. This article critiques these three arguments for a right to the death of the fetus.',
 'The rapid rise of international collaborative science has enabled access to genomic data. In this article, it is argued that to move beyond mapping genomic variation to understanding its role in complex disease aetiol

In [ ]:
import re

def clean_abstract(text):
    if pd.isna(text):
        return None
    # Remove leading 'Abstract', 'ABSTRACT', possibly attached to first word
    text = re.sub(r'^\s*abstract\s*', '', text, flags=re.I)
    return text.strip()

# Apply to your abstracts
df['clean_abstract'] = df['abstract'].apply(clean_abstract)


# Keep only rows with abstracts
df = df[df['clean_abstract'].notna()]

texts = df['clean_abstract'].tolist()
print(f"Number of abstracts available for modeling: {len(texts)}")
texts[:3]  # preview first 3 abstracts



Number of abstracts available for modeling: 424


['Both many critics of abortion and many defenders of abortion have suggested that artificial wombs could end the abortion debate. If the fetus is removed from the uterus, women have an end to an unwanted pregnancy. If the living fetus is then put in an artificial uterus for ectogenesis, there is no termination of the life of the fetus. Joona Räsänen challenges this view in his article, Ectogenesis, abortion and a right to the death of the fetus. Räsänen provides three arguments for a right to secure the death of the human being in utero, namely the ‘right not to become a biological parent argument’, the ‘right to genetic privacy argument’, and the ‘right to property argument’. This article critiques these three arguments for a right to the death of the fetus.',
 'The rapid rise of international collaborative science has enabled access to genomic data. In this article, it is argued that to move beyond mapping genomic variation to understanding its role in complex disease aetiology and 

In [104]:
texts2 = df[df['Enhancement'] > 0]['clean_abstract']

len(texts2)

texts2.to_csv("enhance_abstracts.csv", index=False)


In [100]:
# Optional: add domain-specific stopwords
extra_stopwords = ["study", "result", "method", "participant", "conclusion"]

vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=5,
    stop_words="english",
    ngram_range=(1, 2)
)

# Combine default + extra stopwords
vectorizer.stop_words_ = set(vectorizer.get_stop_words()).union(extra_stopwords)

X = vectorizer.fit_transform(texts2)
print(f"TF-IDF matrix shape: {X.shape}")


TF-IDF matrix shape: (145, 514)


In [ ]:
writer = pd.ExcelWriter('output.xlsx', engine='xlsxwriter')

In [101]:
n_topics = 10  # start with 10, can adjust later

nmf = NMF(
    n_components=n_topics,
    random_state=42,
    init="nndsvd"
)

W = nmf.fit_transform(X)  # document-topic matrix
H = nmf.components_       # topic-term matrix

print("NMF model fitted!")

NMF model fitted!


In [102]:
feature_names = vectorizer.get_feature_names_out()
n_top_words = 10

for topic_idx, topic in enumerate(H):
    top_terms = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_terms)}")


Topic 0: technologies, enhancement, enhancement technologies, human, human enhancement, use, paper, non, new, arguments
Topic 1: bioenhancement, moral bioenhancement, moral, compulsory, health, public health, public, ought, persson, argument
Topic 2: genetic, children, intelligence, people, genetics, traits, principle, child, selection, disability
Topic 3: status, moral status, persons, moral, beings, post, higher, species, threshold, morally
Topic 4: editing, germline, genome, gene, line, human, ethical, scientific, research, health
Topic 5: harris, normal, ethics, parents, choose, normal human, capacities, disability, sex, reason
Topic 6: enhancements, distinction, medical, justice, treatments, biomedical, genetic, critique, argue, use
Topic 7: human, nature, human nature, beings, life, human beings, transhumanism, dignity, human life, good
Topic 8: rights, genetic, techniques, genetic enhancement, technology, enhancement, personal, justice, information, suggested
Topic 9: moral, mor

In [39]:
dominant_topics = W.argmax(axis=1)
df['dominant_topic'] = dominant_topics



In [41]:
df[df['dominant_topic'] == 1]['clean_abstract']

25      Non‐invasive brain stimulation is used to modu...
71      Per Sundström has totally misunderstood what I...
93      Moral bioenhancement is the potential practice...
142     Responding to several leading ideas from a pap...
146     Ought we to improve our cognitive capacities b...
                              ...                        
1233    This paper demonstrates that accounting for th...
1246    Ingmar Persson and Julian Savulescu argue that...
1272    Humans are morally deficient in a variety of w...
1278    :Several lines of reasoning have been employed...
1303    We outline a number of ethical objections to g...
Name: clean_abstract, Length: 87, dtype: object

In [42]:
concern_keywords = {
    "justice": ["justice", "equity", "inequality", "distributive"],
    "fairness": ["fairness", "bias", "discrimination", "prejudice"],
    "unnaturalness": ["unnatural", "tampering", "artificial", "intervention"],
    "autonomy": ["autonomy", "consent", "choice", "decision-making"]
}

# Count occurrences per abstract
def detect_concerns(text):
    text_lower = text.lower()
    found = {}
    for concern, words in concern_keywords.items():
        found[concern] = any(word in text_lower for word in words)
    return found

df['concerns'] = df['clean_abstract'].apply(detect_concerns)


In [44]:
df[['clean_abstract', 'dominant_topic', 'concerns']].head(10)

,clean_abstract,dominant_topic,concerns
1,Both many critics of abortion and many defende...,7,"{'justice': False, 'fairness': False, 'unnatur..."
2,The rapid rise of international collaborative ...,2,"{'justice': False, 'fairness': False, 'unnatur..."
5,"In recent years, as newborn screening has expa...",7,"{'justice': False, 'fairness': False, 'unnatur..."
7,"In an article of this journal, Perry Hendricks...",0,"{'justice': False, 'fairness': False, 'unnatur..."
9,I review the main models of disability and int...,0,"{'justice': False, 'fairness': False, 'unnatur..."
11,There has been growing concern about the pheno...,5,"{'justice': False, 'fairness': False, 'unnatur..."
15,In vitro gametogenesis (IVG) might offer numer...,6,"{'justice': True, 'fairness': False, 'unnatura..."
16,The principle of the child's right to an open ...,0,"{'justice': False, 'fairness': False, 'unnatur..."
20,The development of human genetics raises a wid...,2,"{'justice': False, 'fairness': False, 'unnatur..."
25,Non‐invasive brain stimulation is used to modu...,1,"{'justice': True, 'fairness': False, 'unnatura..."


In [46]:
from sentence_transformers import SentenceTransformer, util

# Each criterion is a label + description
criteria = {
    "AUTONOMY": "Individuals are autonomous, free to decide over their own lives and bodies.",
    "UTILITY": "It benefits people and society effectively, improving lives, health, or outcomes.",
    "NO_HARM": "It does not harm third parties and is equally safe as traditional methods.",
    "CONSENT": "It modifies the genome of persons (embryos, children) who do not consent.",
    "TED": "It is enhancement and not therapy. Only OK for therapeutic purposes.",
    "NATURAL": "It is unnatural. It alters the natural order or people's constitution.",
    "UNSAFE": "There are unknown consequences, side effects, or long-term risks.",
    "FAIRNESS": "It aggravates inequality, is only accessible to some, or is a form of cheating.",
    "EFFORT": "Making an effort is central; TE is taking the easy way out.",
    "INTRINSIC": "Genetic alteration is inherently wrong without further explanation."
}

# Convert to a list of strings for embeddings
criteria_texts = list(criteria.values())
criteria_labels = list(criteria.keys())




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/ivar/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/ivar/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/ivar/Documents/VSCode/means-matter/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/ivar/Documents/VSCode/means-matter/.venv/lib/python3.9/site-packages/traitlets/config/application.py", lin

In [47]:
criteria_texts

['Individuals are autonomous, free to decide over their own lives and bodies.',
 'It benefits people and society effectively, improving lives, health, or outcomes.',
 'It does not harm third parties and is equally safe as traditional methods.',
 'It modifies the genome of persons (embryos, children) who do not consent.',
 'It is enhancement and not therapy. Only OK for therapeutic purposes.',
 "It is unnatural. It alters the natural order or people's constitution.",
 'There are unknown consequences, side effects, or long-term risks.',
 'It aggravates inequality, is only accessible to some, or is a form of cheating.',
 'Making an effort is central; TE is taking the easy way out.',
 'Genetic alteration is inherently wrong without further explanation.']

In [50]:
abstracts = df['clean_abstract'].dropna().tolist()

# Load pre-trained sentence-transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed abstracts
print("Encoding abstracts...")
abstract_embeddings = model.encode(abstracts, convert_to_tensor=True)

# Embed criteria
print("Encoding criteria...")
criteria_embeddings = model.encode(criteria_texts, convert_to_tensor=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding abstracts...
Encoding criteria...


In [51]:
import torch

# For each abstract, find similarity to each criterion
results = []

for i, abstract_emb in enumerate(tqdm(abstract_embeddings)):
    sims = util.cos_sim(abstract_emb, criteria_embeddings)[0]  # shape: (num_criteria,)
    sims_dict = {label: float(sims[idx]) for idx, label in enumerate(criteria_labels)}
    results.append(sims_dict)

# Add results to dataframe
df_criteria = pd.DataFrame(results)
df = pd.concat([df.reset_index(drop=True), df_criteria], axis=1)

# Preview
df.head()


100%|██████████| 424/424 [00:02<00:00, 191.56it/s]


,Unnamed: 0,author,title,sum,Enhancement,Germline,Ecology,Genetics,Offspring,journal,...,AUTONOMY,UTILITY,NO_HARM,CONSENT,TED,NATURAL,UNSAFE,FAIRNESS,EFFORT,INTRINSIC
0,6581,Christopher Kaczor,Ectogenesis and a right to the death of the pr...,0.151,0.0,0.008,0.000,0.143,0.000,Bioethics,...,0.380140,0.166200,0.171569,0.470052,0.150410,0.322165,0.042435,0.149399,-0.021255,0.341440
1,14586,"Bartha Maria Knoppers, Jennifer R Harris, Paul...",From genomic databases to translation: a call ...,0.151,0.0,0.000,0.000,0.147,0.003,JME,...,0.154341,0.178500,0.048160,0.369744,0.191906,0.059270,0.083522,0.106671,0.026947,0.256907
2,10018,Diane B. Paul,Eugenics Redux: “Reproductive Benefit” as a Ra...,0.151,0.0,0.000,0.000,0.000,0.151,HCR,...,0.226460,0.214523,0.218805,0.459095,0.256182,0.301508,0.212681,0.145541,0.000109,0.390720
3,6806,Joona Räsänen,Against the impairment argument: A reply to He...,0.152,0.0,0.000,0.000,0.000,0.152,Bioethics,...,0.288682,0.160413,0.128461,0.362042,0.186567,0.259728,0.175150,0.271421,0.086327,0.379634
4,5946,CHRISOULA ANDREOU,REHABILITATING HUMAN NATURE,0.152,0.0,0.000,0.122,0.000,0.029,Bioethics,...,0.220167,0.285623,0.101683,0.210682,0.321881,0.120374,0.157128,0.207661,0.073417,0.195307


In [52]:
threshold = 0.45  # cosine similarity threshold for flagging a criterion
for label in criteria_labels:
    df[label + "_flag"] = df[label] > threshold

# Example: show counts
for label in criteria_labels:
    print(f"{label}: {df[label + '_flag'].sum()} abstracts flagged")


AUTONOMY: 2 abstracts flagged
UTILITY: 0 abstracts flagged
NO_HARM: 0 abstracts flagged
CONSENT: 111 abstracts flagged
TED: 8 abstracts flagged
NATURAL: 0 abstracts flagged
UNSAFE: 0 abstracts flagged
FAIRNESS: 0 abstracts flagged
EFFORT: 0 abstracts flagged
INTRINSIC: 79 abstracts flagged


In [53]:
df

,Unnamed: 0,author,title,sum,Enhancement,Germline,Ecology,Genetics,Offspring,journal,...,AUTONOMY_flag,UTILITY_flag,NO_HARM_flag,CONSENT_flag,TED_flag,NATURAL_flag,UNSAFE_flag,FAIRNESS_flag,EFFORT_flag,INTRINSIC_flag
0,6581,Christopher Kaczor,Ectogenesis and a right to the death of the pr...,0.151,0.0,0.008,0.000,0.143,0.000,Bioethics,...,False,False,False,True,False,False,False,False,False,False
1,14586,"Bartha Maria Knoppers, Jennifer R Harris, Paul...",From genomic databases to translation: a call ...,0.151,0.0,0.000,0.000,0.147,0.003,JME,...,False,False,False,False,False,False,False,False,False,False
2,10018,Diane B. Paul,Eugenics Redux: “Reproductive Benefit” as a Ra...,0.151,0.0,0.000,0.000,0.000,0.151,HCR,...,False,False,False,True,False,False,False,False,False,False
3,6806,Joona Räsänen,Against the impairment argument: A reply to He...,0.152,0.0,0.000,0.000,0.000,0.152,Bioethics,...,False,False,False,False,False,False,False,False,False,False
4,5946,CHRISOULA ANDREOU,REHABILITATING HUMAN NATURE,0.152,0.0,0.000,0.122,0.000,0.029,Bioethics,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,16255,"Rachel Horton, Anneke M Lucassen",The moral argument for heritable genome editin...,0.698,0.0,0.163,0.000,0.535,0.000,JME,...,False,False,False,True,False,False,False,False,False,True
420,6757,Robert Ranisch,Germline genome editing versus preimplantation...,0.709,0.0,0.390,0.000,0.000,0.319,Bioethics,...,False,False,False,True,False,False,False,False,False,True
421,5782,JOHN DAVIS,SELECTING POTENTIAL CHILDREN AND UNCONDITIONAL...,0.718,0.0,0.000,0.000,0.034,0.684,Bioethics,...,False,False,False,False,False,False,False,False,False,False
422,6752,Tim Lewens,Blurring the germline: Genome editing and tran...,0.749,0.0,0.550,0.000,0.199,0.000,Bioethics,...,False,False,False,True,False,False,False,False,False,True


In [58]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

# Load your abstracts
texts = df['clean_abstract'].dropna().tolist()


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/ivar/nltk_data...


In [59]:
from collections import defaultdict

word_sentiment = defaultdict(float)

for text in texts:
    words = text.split()  # simple split; can be replaced with better tokenization
    for word in words:
        score = sia.polarity_scores(word)['compound']  # ranges -1 to 1
        word_sentiment[word.lower()] += abs(score)

# Sort words by highest absolute sentiment
sorted_words = sorted(word_sentiment.items(), key=lambda x: x[1], reverse=True)

# Top 20 words
top_words = sorted_words[:20]
print("Top words with highest absolute sentiment:")
for word, score in top_words:
    print(f"{word}: {score:.3f}")


Top words with highest absolute sentiment:
ethical: 82.717
argue: 51.680
argument: 41.177
arguments: 36.171
best: 26.113
good: 25.984
harm: 23.319
no: 22.792
negative: 22.304
better: 21.580
argued: 19.505
argues: 17.945
intelligence: 16.208
value: 15.640
well: 15.299
problem: 15.272
problems: 13.263
important: 13.149
support: 12.861
care: 12.348


In [61]:
top_n_words = 15  # how many words to extract per criterion

# Dictionary to store results
criterion_sentiment_words = {}

for label in criteria_labels:
    # Filter abstracts flagged for this criterion
    abstracts = df[df[label + "_flag"]]['clean_abstract'].dropna().tolist()
    
    # Aggregate absolute sentiment per word
    word_sentiment = defaultdict(float)
    for text in abstracts:
        words = text.split()
        for word in words:
            score = sia.polarity_scores(word)['compound']
            word_sentiment[word.lower()] += abs(score)
    
    # Sort and get top N words
    sorted_words = sorted(word_sentiment.items(), key=lambda x: x[1], reverse=True)
    criterion_sentiment_words[label] = sorted_words[:top_n_words]


In [62]:
criterion_sentiment_words

{'AUTONOMY': [('problem', 1.6076),
  ('wronged', 1.3212000000000002),
  ('solution', 0.9545999999999999),
  ('problem,', 0.8038),
  ('best', 0.6369),
  ('meaningful', 0.6364),
  ('undermine', 0.592),
  ('no', 0.592),
  ('harming,', 0.5574),
  ('harming', 0.5574),
  ('rob', 0.5574),
  ('harm', 0.5423),
  ('violating', 0.5423),
  ('wrong', 0.4767),
  ('problem.', 0.4019)],
 'UTILITY': [],
 'NO_HARM': [],
 'CONSENT': [('ethical', 35.23139999999999),
  ('argue', 14.279999999999996),
  ('argument', 8.668800000000003),
  ('harm', 8.1345),
  ('arguments', 8.038000000000004),
  ('negative', 6.862800000000001),
  ('no', 6.808000000000002),
  ('argues', 5.345200000000001),
  ('good', 5.284800000000001),
  ('value', 5.099999999999999),
  ('desire', 4.822800000000001),
  ('important', 4.652900000000002),
  ('risks', 4.371200000000001),
  ('consent', 4.299700000000001),
  ('justice', 4.2136)],
 'TED': [('ethical', 2.5530000000000004),
  ('better', 1.3212000000000002),
  ('great', 1.2498),
  ('mista

In [65]:
import pandas as pd
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import hdbscan

# Load abstracts
abstracts = df['clean_abstract'].dropna().tolist()
print(f"Number of abstracts: {len(abstracts)}")


Number of abstracts: 424


In [66]:
# KeyBERT uses embeddings under the hood
kw_model = KeyBERT('all-MiniLM-L6-v2')

# Optional: for semantic clustering later
embedder = SentenceTransformer('all-MiniLM-L6-v2')
abstract_embeddings = embedder.encode(abstracts, convert_to_tensor=True)

In [79]:
# Parameters
top_n = 5
ngram_range = (1,3)  # unigrams, bigrams, trigrams
stop_words = 'english'

all_phrases = []

keywords = kw_model.extract_keywords(
    abstracts,             # pass the full list, not single abstracts
    keyphrase_ngram_range=(1,3),
    stop_words='english',
    top_n=5,
    use_mmr=True,
    diversity=0.7
)


RuntimeError: Numpy is not available

In [75]:
pip install numpy==1.23.5 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 10.0 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
Note: you may need to restart the kernel to use updated packages.


In [78]:
import numpy as np
print(np.__version__)


2.0.2


In [ ]:
from collections import Counter

phrase_counts = Counter(all_phrases)

# Top 30 emerging phrases (bottom-up concerns)
top_phrases = phrase_counts.most_common(30)
print("Top bottom-up concern phrases:")
for phrase, count in top_phrases:
    print(f"{phrase}: {count}")
